In [ ]:
# Load a expression from json
from asmblr.base import BaseNode
import os
import json
from migumi.utils.converter import fix_format, get_expr_and_state, fix_expr_dict
import migumi.shader
from migumi.shader.compiler import compile_set
from migumi.shader.compile_multipass import compile_set_multipass

from geolipi.torch_compute.sketcher import Sketcher
res = 1024
sketcher_2d = Sketcher(n_dims=2, resolution=res, device='cuda')
sketcher_3d = Sketcher(n_dims=3, resolution=256, device='cuda')
uniforms = {}
# resolved_expr = map_state_to_expr(expr_dict, sketcher_3d, state_map, uniforms)
design_dir = "/users/aganesh8/data/aganesh8/projects/kigumi/migumi-dataset/joints/"

# cur_design = "three_cubes.json"
cur_design = "CJ_AKT/polyline_files/base.json"
# cur_design ="CJ_DT.json"
cur_design_file = os.path.join(design_dir, cur_design)

design_name = cur_design.split(".")[0]


In [ ]:

data =  json.load(open(cur_design_file, "r"))
main_key = list(data['moduleList'].keys())[0]
data = data['moduleList'][main_key]

corrected_data = fix_format(data)
expressions = BaseNode.from_dict(data)
if not isinstance(expressions, list):
    expressions = [expressions]
expr_dict, state_map = get_expr_and_state(expressions)
expr_dict = fix_expr_dict(expr_dict, mode="v4", add_bounding=False)

In [ ]:

import geolipi.symbolic as gls
import numpy as np
from sysl.shader.shader_module import SMMap
from IPython.display import display, HTML
from sysl.shader_runtime.generate_shader_html import create_shader_html, make_jupyter_compatible_html
from sysl.shader_runtime.generate_shader_html import create_multibuffer_shader_html
from sysl.shader_runtime.offline_render import render_multipass
settings = {
    "render_mode": "v4",
    "variables": {
        "_ADD_FLOOR_PLANE": False,
        "castShadows": True,
        "_AA": 2,
        "_RAYCAST_MAX_STEPS": 200,
        "resolution": (1024, 1024),
        "outline_nhbd": 2,
        "cameraDistance": 5,
        "cameraAngleX": np.pi/4,
        "cameraAngleY": np.pi/4,
        "cameraOrigin": (0, 1, 0),
        "globalStateStep": 1.0,
    },
    "set_to_ubo": False,
    "export_params": False,
    # "target": "ShaderToy",
    # "convert_uniforms_to_constants": True,

}
all_shader_bundles = compile_set_multipass(expr_dict, state_map, settings=settings,
     post_process_shader=["part_outline_nobg"])
for shader_bundle in all_shader_bundles:
    uniforms = shader_bundle['uniforms']
    if "globalStateStep" in uniforms:
        uniforms["globalStateStep"]['init_value'] = 1.0

output = render_multipass(all_shader_bundles)
from PIL import Image
image = Image.fromarray(output)
image